In [ ]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI

# -------- STEP 1: Load PDF --------
pdf_path = "sample.pdf"  # <-- put your PDF here
if not os.path.isfile(pdf_path):
    raise FileNotFoundError(f"PDF file '{pdf_path}' not found. Please place the PDF file in the current directory or update the 'pdf_path' variable.")
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# -------- STEP 2: Chunk Text --------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)

# -------- STEP 3: Embeddings --------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c.page_content for c in chunks]
embeddings = embed_model.encode(texts)

# -------- STEP 4: Create FAISS --------
vector_store = FAISS.from_embeddings(embeddings, texts)

# -------- STEP 5: Ask Question --------
question = input("Enter your question: ")
q_emb = embed_model.encode([question])[0]

# Find similar chunks
docs = vector_store.similarity_search_by_vector(q_emb, k=3)

context = "\n\n".join(docs)

# -------- STEP 6: LLM Answer --------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = f"""
Use the context to answer the question.

Context:

{context}

Question: {question}

Answer:
"""

answer = llm.invoke(prompt).content
print("\nANSWER:", answer)

import sys
print(sys.executable)
